# RT0/P0 Darcy equations and attributed NeoPZ comparisons

The current calculation below assembles the RT0/P0 mixed Darcy equations through
`LocalEquations`, `Equation` and `MultiscaleProblem`. It refines the same affine,
polynomial, cosine and fitted layered physical problems used in the retained
NeoPZ comparison. Pressure data act weakly on exterior faces. The skeletal
unknown is physical normal flux in the declared global face orientation.

The original 46 matched NeoPZ acquisitions are retained as scalar records and
provenance. Their coefficient files and instrumented external driver are not
distributed. Replaying those original fields requires `python -m scripts.run_notebooks /path/to/14_neopz_hdiv_comparison.ipynb --historical`
and their checksum-matching archives; the current solves have their own mesh and
execution provenance. See [the comparison case](https://ipes-lncc.github.io/pymhm/cases/neopz/).

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/eec1e72c2981816909f8f0bff1a3675e60bf4838b4dd2def0d2d03daa410c86d/14_neopz_hdiv_comparison-companion.zip"
COMPANION_SHA256 = "eec1e72c2981816909f8f0bff1a3675e60bf4838b4dd2def0d2d03daa410c86d"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/14_neopz_hdiv_comparison.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


## Declare and execute the physical equations

The local coefficients are flux, pressure and boundary pressure. With physical
divergence matrix D, inverse-permeability mass M and normal extraction N,
the local equation has the following blocks:

$$
\begin{bmatrix}M&-D^T&N\\-D&0&0\\N^T&0&0\end{bmatrix}
\begin{bmatrix}q\\p\\r\end{bmatrix}+B\lambda
=\begin{bmatrix}0\\-f\\0\end{bmatrix}.
$$

The joint constant-pressure kernel and its physical integral are declared
explicitly. The outer weak pressure equation uses C=-B^T. The public element
provider shown below supplies quadrature, basis moments and signed trace maps;
it calls no prepared Darcy solver.


In [ ]:
from functools import partial
import inspect
import os
import json
import numpy as np
from IPython.display import Markdown, display
from examples.analytical_darcy import darcy_fields
from examples.formulations.mixed_darcy import (
    MixedDarcyDefinition, rt_local_equations, recover_rt_darcy,
)
from pymhm import Equation, MultiscaleProblem, TriangleMesh, FaceSpace, SkeletonSpace, assemble
from pymhm.fem.scalar.operators import boundary_data

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
print(inspect.getsource(rt_local_equations))
current = []
for case in ("aff", "poly", "cos", "layer", "layer1000"):
    exact_pressure, exact_flux = darcy_fields(case)
    if case == "cos":
        source = lambda points: 2*np.pi**2*np.cos(np.pi*points[:,0])*np.cos(np.pi*points[:,1])
    else:
        source = -4.0 if case == "poly" else 0.0
    contrast = 1000.0 if case == "layer1000" else 10.0
    permeability = (lambda points, contrast=contrast: np.where(points[:,0] < .5, 1., contrast)) if case.startswith("layer") else 1.0
    for n, segments in ((2, 1), (2, 2), (4, 2), (8, 2)):
        mesh = TriangleMesh.unit_square(n)
        skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(0, segments) for _ in mesh.faces))
        boundary, fixed = boundary_data(skeleton, exact_pressure, order=8)
        provider = partial(rt_local_equations, mesh=mesh, skeleton=skeleton, degree=0,
                           refinement=2, permeability=permeability, source=source, order=8)
        problem = MultiscaleProblem(
            Equation(0, np.r_[boundary, np.zeros(len(mesh.cells))]), provider,
            range(len(mesh.cells)), skeleton.size, (1,)*len(mesh.cells), fixed=fixed,
        )
        system = assemble(problem)
        coefficients = system.solve()
        declaration = MixedDarcyDefinition(problem, skeleton, permeability, source, 0, 8, None)
        result = recover_rt_darcy(declaration, system, coefficients)
        equilibrium = max(float(np.max(abs(value))) for value in result.fine_equilibrium_residuals())
        current.append((case, n, segments, result.l2_error(exact_pressure, 10),
                        result.flux_l2_error(exact_flux, 10), equilibrium))
        assert equilibrium < 1e-10
lines = ["| Physical case | Macro n | Face segments | Pressure L2 error | Flux L2 error | Fine equilibrium |",
         "|---|---:|---:|---:|---:|---:|"]
lines.extend(f"| {case} | {n} | {segments} | {p:.6e} | {q:.6e} | {balance:.3e} |"
             for case, n, segments, p, q, balance in current)
display(Markdown("\n".join(lines)))
report = json.loads((ROOT / "examples/results/neopz/comparison.json").read_text())
print("Retained original comparison provenance:", report.get("reference", report.get("revisions")))
print("Archived matched comparisons:", len(report["rows"]))


## Optional replay of the original comparison fields

These cells require the original externally acquired coefficient archives and
published figures. Checksums identify the recorded acquisitions; newly solved
fields do not replace those coefficient vectors.

In [ ]:
if HISTORICAL_REPLAY:
    from pathlib import Path
    import hashlib
    import json
    
    from IPython.display import Image, Markdown, display
    
    RESULTS = ROOT / "examples/results/neopz"
    report = json.loads((RESULTS / "comparison.json").read_text())
    rows = report["rows"]
    for row in rows:
        path = RESULTS / row["archive"]
        assert hashlib.sha256(path.read_bytes()).hexdigest() == row["archive_sha256"], path.name
    print(f"Loaded {len(rows)} records and verified all field archive checksums")
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/14_neopz_hdiv_comparison.ipynb --historical with the attributed archives.")


## Analytical problem data

For the affine patch, pressure is p=1+x+2y, permeability is one, source is zero, and physical flux is q=(-1,-2). The mixed pressure is cellwise constant, so its exact representable target is the cell mean, not the pointwise analytical pressure.

The layered cases have permeability contrast 10 or 1000, continuous normal flux -1 and a tangential flux jump from -2 to -2k. That jump is physically admissible in H(div).


In [ ]:
if HISTORICAL_REPLAY:
    for case in sorted({row["case"] for row in rows}):
        records = [row for row in rows if row["case"] == case]
        levels = sorted({row["macro_resolution"] for row in records})
        traces = sorted({row["trace_segments"] for row in records})
        print(f"{case}: n={levels}, trace segments={traces}, {len(records)} archived results")
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/14_neopz_hdiv_comparison.ipynb --historical with the attributed archives.")


## Recorded field differences

The table displays the measurements already obtained at n=2. During the executed study, physical pressure, flux and divergence fields were compared on matched meshes. The stored three-point triangle rule integrates squared RT0 field differences exactly; errors against smooth analytical fields use higher-order quadrature.

Displaying the measurements and checking archive integrity does not constitute a new solver execution.


In [ ]:
if HISTORICAL_REPLAY:
    small_rows = [row for row in rows if row["macro_resolution"] == 2]
    header = "| Case | Trace segments | Pressure difference L2 | Flux difference L2 | Divergence difference L2 |"
    lines = [header, "|---|---:|---:|---:|---:|"]
    for result in small_rows:
        lines.append(
            f"| {result['case']} | {result['trace_segments']} | "
            f"{result['pressure_difference_l2']:.3e} | "
            f"{result['flux_difference_l2']:.3e} | "
            f"{result['divergence_difference_l2']:.3e} |"
        )
    display(Markdown("\n".join(lines)))
    print(f"Displayed {len(small_rows)} previously executed comparisons")
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/14_neopz_hdiv_comparison.ipynb --historical with the attributed archives.")


## Five-level convergence and the actual macro mesh

The error curves compare both implementations for full and restricted traces. The field figures show analytical, NeoPZ, pyMHM, and signed implementation-difference panels. The first three columns share physical color scales; differences have their own scales.

Every field panel contains the actual macro mesh. Flux is rendered affinely inside each fine triangle without smoothing jumps; pressure remains P0. Analytical cosine fields are sampled for display only, independently of the quadrature used in error norms.

In [ ]:
if HISTORICAL_REPLAY:
    figures = ROOT / "docs/figures/neopz"
    for filename in ("darcy-convergence.png", "cos-fields-n8-s1.png"):
        display(Image(filename=str(figures / filename)))
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/14_neopz_hdiv_comparison.ipynb --historical with the attributed archives.")


## Fitted material interfaces

Compare the constant normal flux with the tangential jump at x=0.5. The profile uses y=0.37 and marks all crossings of the actual macro mesh. Its P0 pressure steps are the computed cell values, not an interpolated pressure reconstruction.

In [ ]:
if HISTORICAL_REPLAY:
    for filename in (
        "layer-fields-n4-s1.png",
        "layer1000-fields-n4-s1.png",
        "layer-profiles.png",
    ):
        display(Image(filename=str(figures / filename)))
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/14_neopz_hdiv_comparison.ipynb --historical with the attributed archives.")


To regenerate these figures from the archived data, run:

    python -m scripts.run_notebooks /path/to/14_neopz_hdiv_comparison.ipynb --study

The recorded restricted reference used T-transpose A T from NeoPZ's mixed operator, with matching normal orientations and edge-integral scaling. Its fields were evaluated by NeoPZ after the restricted SciPy solve. The record checks both the local RT0 operators and the reduced MHM trace; this notebook visualizes that completed study.
